# Create slides with the Responses API and GPT Image

This notebook creates a short business slide from a JSON dataset. It uses the Responses API with Code Interpreter for analysis, then uses the image-generation tool and `python-pptx` to assemble a presentation.


## Prerequisites

Set `OPENAI_API_KEY` in your environment. This notebook makes billable API calls and writes `notrealcorp_quarterly_review.pptx` to the current directory.


In [ ]:
%pip install --upgrade openai pandas python-pptx pillow

import base64
import os
from io import BytesIO

import pandas as pd
from IPython.display import Image, display
from openai import OpenAI
from pptx import Presentation
from pptx.util import Inches

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o"
DATA_PATH = "data/NotRealCorp_financial_data.json"


## Upload the dataset and create an analysis

Attach the uploaded file to an automatically created Code Interpreter container. The response includes generated text and structured Code Interpreter output.


In [ ]:
financial_data = pd.read_json(DATA_PATH)
financial_data.head()

with open(DATA_PATH, "rb") as dataset:
    uploaded_file = client.files.create(file=dataset, purpose="user_data")

analysis_prompt = (
    "Calculate profit (revenue minus cost) by quarter and year. Create a line plot across "
    "distribution channels using green, light red, and light blue lines. Then summarize the "
    "most decision-relevant findings for an executive audience."
)

analysis_response = client.responses.create(
    model=MODEL,
    input=analysis_prompt,
    tools=[{
        "type": "code_interpreter",
        "container": {"type": "auto", "file_ids": [uploaded_file.id]},
    }],
)

print(analysis_response.output_text)


## Display the generated chart

Code Interpreter image outputs expose a URL. Save the first image locally so it can be added to the presentation.


In [ ]:
code_call = next(
    item for item in analysis_response.output if item.type == "code_interpreter_call"
)
chart = next(output for output in code_call.outputs or [] if output.type == "image")
display(Image(url=chart.url))

import requests

chart_path = "quarterly_profit.png"
with open(chart_path, "wb") as image_file:
    image_file.write(requests.get(chart.url, timeout=30).content)


## Turn the analysis into slide copy

Continue the same analytical context with `previous_response_id`, then request a concise title and two supporting bullets.


In [ ]:
copy_response = client.responses.create(
    model=MODEL,
    previous_response_id=analysis_response.id,
    input=(
        "Write a slide title and exactly two executive bullets based on the analysis. "
        "Each bullet should be 20-30 words and explain the business implication."
    ),
)

slide_copy = copy_response.output_text
print(slide_copy)


## Generate a supporting image

The Responses image-generation tool returns base64-encoded image data in an `image_generation_call` output item.


In [ ]:
image_response = client.responses.create(
    model=MODEL,
    input=(
        "Create a clean, editorial-style image that conveys sustainable business growth "
        "and operational momentum. It will appear beside a quarterly-profit chart."
    ),
    tools=[{"type": "image_generation"}],
)

image_call = next(
    item for item in image_response.output if item.type == "image_generation_call"
)
hero_image = base64.b64decode(image_call.result)
hero_path = "growth_hero.png"
with open(hero_path, "wb") as image_file:
    image_file.write(hero_image)

display(Image(data=hero_image))


## Assemble the slide

Use `python-pptx` to combine the generated chart, image, and slide copy in a PowerPoint file.


In [ ]:
presentation = Presentation()
slide = presentation.slides.add_slide(presentation.slide_layouts[5])

title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.25), Inches(12.0), Inches(1.0))
title_box.text_frame.text = "Quarterly profit review"

copy_box = slide.shapes.add_textbox(Inches(0.5), Inches(1.2), Inches(5.5), Inches(2.5))
copy_box.text_frame.text = slide_copy

slide.shapes.add_picture(chart_path, Inches(0.5), Inches(4.0), width=Inches(6.0))
slide.shapes.add_picture(hero_path, Inches(7.0), Inches(1.2), width=Inches(5.5))

output_path = "notrealcorp_quarterly_review.pptx"
presentation.save(output_path)
print(f"Wrote {output_path}")


## Clean up

Delete uploaded files when you no longer need them. This notebook does not create an Assistant, Thread, or Run.


In [ ]:
client.files.delete(uploaded_file.id)
